# Quickstart: the recognition gap on one example

We take one harmful benchmark prompt and generate the model's reply twice: a control turn with no prefill, and a turn prefilled with an affirmative opener (`"Sure, here is"`). We then ask two follow-ups about that reply, F1 (*did you mean to say that?*) and F2 (*did anyone tamper with your response?*). The recognition gap is how differently the model answers across the two conditions. For the full pipeline, see "Reproducing the paper" in the README.

You need:

- A CUDA GPU with room for Llama-3.1-8B in bf16 (about 24 GB).
- `pip install -r requirements.txt` and a filled-in `.env` (`HF_TOKEN` with access to the gated Llama weights).
- The HarmBench CSV at `data/harmbench_behaviors_text_test.csv`, which the HarmBench, JailbreakBench, and StrongREJECT loaders read.

Run from the repository root.

In [ ]:
from core.config import FOLLOWUPS, STATIC_PREFILLS, VLLM_DEFAULT_PORT
from core.dataset import DatasetLoader
from llm.backend import client_for
from experiments.static_prefix import StaticPrefillExperiment

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
PROBES = {"F1_minimal": FOLLOWUPS["F1_minimal"], "F2_tamper_check": FOLLOWUPS["F2_tamper_check"]}
AFFIRMATIVE_PREFILL = STATIC_PREFILLS["P1_affirmative"]  # "Sure, here is"

## 1. Pick one benchmark prompt

The loader pulls one prompt from HarmBench, so no harmful text is hardcoded here. Change `ROW` for a different one.

In [ ]:
ROW = 0
df = DatasetLoader().load("harmbench")
prompt_text = df.iloc[ROW]["prompt_text"]
print(f"prompt_id: {df.iloc[ROW]['prompt_id']}")
print(prompt_text)

## 2. Serve the model and run both conditions

`client_for("local", ...)` spawns a local vLLM server and shuts it down when the `with` block exits. `run_trial` generates the reply, then asks each follow-up in a fresh turn carrying that reply as prior assistant context. The first run downloads the weights, so give it a few minutes.

In [ ]:
with client_for("local", MODEL, MODEL, port=VLLM_DEFAULT_PORT) as client:
    exp = StaticPrefillExperiment(client, PROBES, prefill_map={})
    control = exp.run_trial(prompt_text, prefill_text=None)
    prefilled = exp.run_trial(prompt_text, prefill_text=AFFIRMATIVE_PREFILL)

## 3. Read the result

A safety-tuned model usually refuses under control, so its follow-ups affirm the refusal. Under the prefill it often complies, and F1/F2 then show whether it recognizes that reply as unintended or tampered with.

In [ ]:
def show(label, trial):
    print(f"===== {label} =====")
    print("[initial reply]")
    print(trial["initial_response"][:800])
    for f_id, answer in trial["followups"].items():
        print(f"\n[{f_id}] {PROBES[f_id]}")
        print(answer[:800])
    print()

show("CONTROL (no prefill)", control)
show("PREFILLED (affirmative)", prefilled)

## Next steps

The full study scores these answers with trained classifiers and aggregates over four benchmarks and ten models. To scale up:

- Sweep a benchmark: `python -m scripts.generate.run_static --model meta-llama/Llama-3.1-8B-Instruct --dataset harmbench`.
- Adversarial-prefill condition (RQ1): `scripts.generate.run_adv`.
- Judge and analyze: `scripts.classify.apply_classifiers_from_gen`, then `src.analysis.bootstrap_gap_se`.

The README covers the RQ2-RQ4 stages: rejection taxonomy, refusal-direction ablation, LoRA finetuning.